# Plot and interpret · Solutions

Pandas, Clearly · Pandas for Data Science · Version 1.0.0

Dr. Ali Al Bataineh  
Computer Information Science  
Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates

Read, predict, run, change, and explain. Every worked example starts from fresh teaching data. The original student/café tables are fictional; the public Penguins files retain their documented source.

Run the setup cell first. Keep this notebook inside the downloaded course folder so its `data/` folder can be found. Exports go to `outputs/`. For public-data lessons, extract the full notebook practice pack.

Questions: What does one row represent? Which files are inputs, and where will your outputs be written?

In [ ]:
# Pandas, Clearly
# Dr. Ali Al Bataineh
# Computer Information Science
# Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates
# Run once before the lessons. It prepares explicitly fictional practice data.
# Questions: What does 'as pd' mean? Where will exported files be saved?
# Try: print(pd.__version__) and print(OUTPUT_DIR.resolve()).
# Explain: Why should source data and analysis outputs use different folders?
import os
import tempfile
from pathlib import Path
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "pandas-clearly-mpl"))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import sqlite3
from pathlib import Path
from io import StringIO

# Find data from the repository root or either notebook subdirectory.
# Browser workers mount the same data tree at /course/data.
COURSE_ROOT = next(
    (parent for parent in [Path.cwd(), *Path.cwd().parents]
     if (parent / "data" / "students.csv").is_file()),
    Path.cwd(),
)
DATA_DIR = COURSE_ROOT / "data"
OUTPUT_DIR = COURSE_ROOT / "outputs"
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

_STUDENT_ROWS = [
    ["S001", "Amira", "RAK", "AI", 20, 8, 88, 95, "2026-09-01"],
    ["S002", "Bilal", "Dubai", "IT", 21, 5, 72, 85, "2026-09-02"],
    ["S003", "Chloe", "RAK", "AI", 19, 9, 91, 98, "2026-09-03"],
    ["S004", "Dana", "Dubai", "IT", 22, 4, 65, 78, "2026-09-04"],
    ["S005", "Ethan", "RAK", "IT", 20, 6, 79, 90, "2026-09-05"],
    ["S006", "Fatima", "Dubai", "AI", 21, 10, 95, 99, "2026-09-06"],
    ["S007", "Grace", "RAK", "AI", 23, 7, 84, 92, "2026-09-07"],
    ["S008", "Hassan", "Dubai", "IT", 20, 3, 58, 70, "2026-09-08"],
    ["S009", "Ivy", "RAK", "IT", 22, 6, 76, 88, "2026-09-09"],
    ["S010", "Jamal", "Dubai", "AI", 19, 8, 89, 96, "2026-09-10"],
    ["S011", "Khalid", "RAK", "AI", 24, 5, 68, 82, "2026-09-11"],
    ["S012", "Lina", "Dubai", "IT", 21, 7, 82, 94, "2026-09-12"],
]
_STUDENT_COLUMNS = ["Student_ID", "Name", "Campus", "Program", "Age",
                    "Study_Hours", "Score", "Attendance", "Join_Date"]

def students():
    """Return a new, clean practice DataFrame, never real student records."""
    return pd.DataFrame(_STUDENT_ROWS, columns=_STUDENT_COLUMNS)

def records():
    """Return the same practice records as a list of dictionaries."""
    return students().to_dict(orient="records")

def raw_students():
    """Return a fresh table with deliberate data-quality problems."""
    df = students()
    df["Score"] = df["Score"].astype(object)
    df["Study_Hours"] = df["Study_Hours"].astype(object)
    df.loc[0, "Name"] = " Amira "
    df.loc[1, "Campus"] = "dubai"
    df.loc[3, "Score"] = None
    df.loc[5, "Study_Hours"] = None
    df.loc[7, "Score"] = "absent"
    df.loc[10, "Attendance"] = 105
    return pd.concat([df, df.iloc[[2]]], ignore_index=True)

def departments():
    return pd.DataFrame({
        "Program": ["AI", "IT"],
        "Department": ["Computing", "Information Systems"],
        "Advisor": ["Dr. Noor", "Dr. Sami"],
    })

def assessment():
    return pd.DataFrame({
        "Student_ID": ["S001", "S002", "S003", "S004", "S005", "S006",
                       "S007", "S008", "S009", "S010", "S011", "S013"],
        "Quiz": [17, 14, 19, 13, 16, 19, 17, 11, 15, 18, 14, 16],
        "Project": [44, 36, 46, 34, 40, 47, 42, 30, 38, 45, 35, 41],
    })

def attendance():
    return pd.DataFrame({
        "Date": ["2026-10-01", "2026-10-02", "2026-10-03",
                 "2026-10-05", "2026-10-06", "2026-10-07"],
        "Visitors": [20, 24, 18, 30, 28, 32],
        "Campus": ["RAK"] * 6,
    })

# Create missing practice files. Existing files are preserved.
for _name, _frame in [
    ("students.csv", students()), ("raw_students.csv", raw_students()),
    ("departments.csv", departments()), ("assessment.csv", assessment()),
    ("attendance.csv", attendance()),
]:
    if not (DATA_DIR / _name).exists():
        _frame.to_csv(DATA_DIR / _name, index=False)
if not (DATA_DIR / "students.json").exists():
    students().to_json(DATA_DIR / "students.json", orient="records", indent=2)
if not (DATA_DIR / "students.xlsx").exists():
    try:
        students().to_excel(DATA_DIR / "students.xlsx", sheet_name="Students", index=False)
    except ImportError:
        print("Excel needs openpyxl. Install it before Module 20's Excel cell.")

pd.set_option("display.max_rows", 15)
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 110)
plt.rcParams.update({"figure.figsize": (7, 4), "axes.spines.top": False,
                     "axes.spines.right": False, "figure.dpi": 110})
print("Ready. pandas", pd.__version__)
print("Practice data: 12 fictional students. Every helper returns a fresh table.")
print("Exports go to:", OUTPUT_DIR.resolve())


## Module 24. Plot and interpret

Select a plot that answers a clear question and label its units.

Before you start: Use summaries, numeric columns, and sorted dates.

Official reference: https://pandas.pydata.org/docs/user_guide/visualization.html

### M24C01. Compare groups with bars

A bar chart compares categories. First calculate one statistic per group, then plot those values. The axis label must say whether the height is a mean, count, or total.

**Read:** What does the height of one bar represent?

Answer: The mean student Score for that campus in the fictional table.

**Predict:** Which campus has the higher mean?

Answer: RAK, with 81.00, compared with Dubai at about 76.83.

**Explain:** Does the chart prove campus causes a score difference?

Answer: No. These are descriptive summaries from a small fictional dataset, without causal evidence.



In [ ]:
# Predict before running: Which campus has the higher mean?
df = students()
result = df.groupby("Campus")["Score"].mean().sort_index()
ax = result.plot.bar(color="#28756a", rot=0, figsize=(6, 3.6))
ax.set(title="Mean score by campus (practice data)", xlabel="Campus", ylabel="Mean score (out of 100)", ylim=(0, 100))
plt.tight_layout()
print(result.round(2))
plt.show()

**Your turn:** Plot the number of students in each Program instead, with a count label on the y-axis.

For the check, keep these result names: `result`.

**Hint:** Use groupby("Program").size() before plot.bar().

**Watch for this:** Use a zero baseline for ordinary bar comparisons so bar length remains meaningful.

In [ ]:
# Change this code to complete the task.
# Question: Plot the number of students in each Program instead, with a count label on the y-axis.
df = students()
result = df.groupby("Program").size()
ax = result.plot.bar(color="#28756a", rot=0, figsize=(6, 3.6))
ax.set(title="Students by program (practice data)", xlabel="Program", ylabel="Number of students")
ax.set_ylim(bottom=0)
plt.tight_layout()
print(result)
plt.show()

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result.to_dict() == {"AI": 6, "IT": 6}
print("Code checks passed. Review the written explanation as well.")

Both programs have six students, so their bars have the same height.

**Quick check:** Why label the y-axis "Mean score"?

1. To identify the statistic and units
2. To make the bar taller
3. To remove uncertainty

Answer: 1. A viewer needs to know what each plotted quantity means.

**Explain what you learned:** Aggregate first, then label exactly what was plotted.

### M24C02. Inspect a distribution

A histogram counts numeric observations within intervals called bins. It shows the distribution of individual values, rather than a summary for each named category. Bin choices can change the visual impression.

**Read:** What is counted along the vertical axis?

Answer: Students whose scores fall inside each interval.

**Predict:** How many scores are in [90, 100)?

Answer: 2: 91 and 95.

**Explain:** How is this different from a bar chart of campus means?

Answer: A histogram groups numeric values into intervals; campus bars compare named groups.



In [ ]:
# Predict before running: How many scores are in [90, 100)?
df = students()
result = pd.cut(df["Score"], bins=[50, 60, 70, 80, 90, 100], right=False).value_counts(sort=False)
ax = df["Score"].plot.hist(bins=[50, 60, 70, 80, 90, 100], color="#28756a", edgecolor="white", figsize=(6, 3.6))
ax.set(title="Distribution of scores (practice data)", xlabel="Score (out of 100)", ylabel="Number of students")
plt.tight_layout()
print(result)
plt.show()

**Your turn:** Use wider bins [50, 75, 100] and compare the count table and visual shape.

For the check, keep these result names: `result`.

**Hint:** Change the bin edges in both pd.cut and plot.hist.

**Watch for this:** Histogram libraries usually include the upper endpoint of the last bin; pd.cut(right=False) does not. These sample scores contain no endpoint value of 100.

In [ ]:
# Change this code to complete the task.
# Question: Use wider bins [50, 75, 100] and compare the count table and visual shape.
df = students()
result = pd.cut(df["Score"], bins=[50, 75, 100], right=False).value_counts(sort=False)
ax = df["Score"].plot.hist(bins=[50, 75, 100], color="#28756a", edgecolor="white", figsize=(6, 3.6))
ax.set(title="Scores with wider bins (practice data)", xlabel="Score (out of 100)", ylabel="Number of students")
plt.tight_layout()
print(result)
plt.show()

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result.tolist() == [4, 8]
assert result.sum() == 12
print("Code checks passed. Review the written explanation as well.")

The two wider bins contain four scores below 75 and eight scores from 75 up to 100.

**Quick check:** What happens when bin widths change?

1. The raw scores change
2. The grouped visual summary changes
3. The number of students changes

Answer: 2. Binning changes the display, not the underlying observations.

**Explain what you learned:** Check bin boundaries when interpreting distributions.

### M24C03. Compare two measurements

A scatter plot places one point at each pair of numeric measurements. This can reveal a pattern, an unusual observation, or a gap. Correlation describes association; it does not establish why the pattern occurs.

**Read:** What does one dot represent?

Answer: One student with a Study_Hours value and a Score value.

**Predict:** Will the overall pattern slope upward or downward?

Answer: Upward in this fictional table: higher study hours generally accompany higher scores.

**Explain:** Can this prove that adding one hour causes a fixed score increase?

Answer: No. The table is fictional and descriptive, and the plot does not control other factors.



In [ ]:
# Predict before running: Will the overall pattern slope upward or downward?
df = students()
result = df[["Study_Hours", "Score"]].corr().loc["Study_Hours", "Score"]
ax = df.plot.scatter(x="Study_Hours", y="Score", color="#28756a", figsize=(6, 3.6))
ax.set(title="Study time and score (practice data)", xlabel="Study hours per week", ylabel="Score (out of 100)")
plt.tight_layout()
print("Pearson correlation:", round(result, 3))
plt.show()

**Your turn:** Plot Attendance against Score and calculate their Pearson correlation.

For the check, keep these result names: `result`.

**Hint:** Replace Study_Hours with Attendance in both the correlation and plot.

**Watch for this:** Outliers, restricted ranges, and missing data can affect correlation; inspect the actual points.

In [ ]:
# Change this code to complete the task.
# Question: Plot Attendance against Score and calculate their Pearson correlation.
df = students()
result = df[["Attendance", "Score"]].corr().loc["Attendance", "Score"]
ax = df.plot.scatter(x="Attendance", y="Score", color="#28756a", figsize=(6, 3.6))
ax.set(title="Attendance and score (practice data)", xlabel="Attendance (%)", ylabel="Score (out of 100)")
plt.tight_layout()
print("Pearson correlation:", round(result, 3))
plt.show()

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert 0 < result <= 1
assert np.isclose(result, students()["Attendance"].corr(students()["Score"]))
print("Code checks passed. Review the written explanation as well.")

The revised plot compares attendance percentages with scores. Its association still does not establish causation.

**Quick check:** What does a positive correlation indicate?

1. One variable must cause the other
2. Values tend to increase together
3. Every point lies on a straight line

Answer: 2. A positive correlation describes the direction of association, not a causal mechanism.

**Explain what you learned:** Use scatter plots to inspect relationships without claiming causes.

### M24C04. Show change over time

A line chart uses an ordered horizontal axis to show change. Reindexing to a daily grid keeps the missing date visible as a gap. A line across an unobserved day can otherwise suggest continuity that was not measured.

**Read:** Why convert Date before plotting?

Answer: It gives the horizontal axis chronological meaning and supports a regular daily index.

**Predict:** Where will the gap appear?

Answer: October 4, whose visitor count is missing.

**Explain:** Does the gap mean no visitors came?

Answer: No. The count is unknown for that day.



In [ ]:
# Predict before running: Where will the gap appear?
df = attendance()
df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m-%d")
result = df.set_index("Date")["Visitors"].sort_index().asfreq("D")
fig, ax = plt.subplots(figsize=(6.5, 3.6))
ax.plot(result.index, result, marker="o", color="#28756a")
ax.set(title="Daily visitors (practice data)", xlabel="Date", ylabel="Recorded visitors")
ax.set_ylim(bottom=0)
fig.autofmt_xdate()
plt.tight_layout()
print(result)
plt.show()

**Your turn:** Store the daily counts in daily and the three-day rolling mean in result, requiring three known daily values. Plot daily and result together with labels for both lines.

For the check, keep these result names: `result`.

**Hint:** Calculate result from daily.rolling(3, min_periods=3).mean(), then plot both Series on the same axes.

**Watch for this:** A smooth line does not create new observations; report how any smoothing was calculated.

In [ ]:
# Change this code to complete the task.
# Question: Store the daily counts in daily and the three-day rolling mean in result, requiring three known daily values. Plot daily and result together with labels for both lines.
df = attendance()
df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m-%d")
daily = df.set_index("Date")["Visitors"].sort_index().asfreq("D")
result = daily.rolling(3, min_periods=3).mean()
fig, ax = plt.subplots(figsize=(6.5, 3.6))
ax.plot(daily.index, daily, marker="o", label="Daily count", color="#28756a")
ax.plot(result.index, result, marker="s", label="3-day mean", color="#9c6b28")
ax.set(title="Daily visitors and rolling mean (practice data)", xlabel="Date", ylabel="Visitors")
ax.set_ylim(bottom=0)
ax.legend()
fig.autofmt_xdate()
plt.tight_layout()
print(result.round(2))
plt.show()

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result.notna().sum() == 2
assert result.loc["2026-10-07"] == 30
print("Code checks passed. Review the written explanation as well.")

The rolling mean is available only on October 3 and October 7 because other windows lack three known values.

**Quick check:** Why leave a visible gap for the missing day?

1. It communicates an unknown measurement
2. It proves the count was zero
3. It increases the sample size

Answer: 1. A gap prevents an unknown value from being mistaken for an observed zero.

**Explain what you learned:** Time order, gaps, and transformations belong in the explanation.

## Keep practicing

Can you apply this idea to a different table without copying the example?

Dr. Ali Al Bataineh  
Computer Information Science  
Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates